# 🫀 Cleveland Heart Disease — Model Training Pipeline
### Minor Project Part-I: Model Architecture & Fitting
**Dataset:** Cleveland Heart Disease (`processed_data/cleaned_heart_disease.csv`)  
**Scope:** **Task 3 — Training Only (No Evaluation)**  
> **Note:** As specified in Task 3, this notebook strictly focuses on **data partitioning, feature scaling, model instantiation, fitting on training data, and model serialization**. No evaluation metrics, confusion matrices, or test set scoring are performed in this phase.

## 1. Setup & Library Imports
Import machine learning estimators from `scikit-learn` and utilities for model persistence.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Scikit-learn estimators & preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Directory paths
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "cleaned_heart_disease.csv")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Environment & libraries initialized successfully.")
print(f"Model Artifacts will be saved to: {MODEL_DIR}")

Environment & libraries initialized successfully.
Model Artifacts will be saved to: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models


## 2. Data Ingestion & Train-Test Split
Load the verified cleaned dataset and perform an 80/20 stratified split to isolate training data ($N_{\text{train}} = 820$, $N_{\text{test}} = 205$).

In [2]:
# 2.1 Load Cleaned Dataset
df = pd.read_csv(DATA_PATH)
print(f"Loaded Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

X = df.drop(columns=['target'])
y = df['target']

# 2.2 Stratified 80/20 Train-Test Partitioning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=23, stratify=y
)

print(f"Training Features Shape (X_train) : {X_train.shape}")
print(f"Training Labels Shape (y_train)   : {y_train.shape}")
print(f"Testing Features Shape (X_test)   : {X_test.shape} (Preserved untouched for Phase 4)")
print(f"Testing Labels Shape (y_test)     : {y_test.shape}")
print("\nTarget distribution in Training Set:")
print(y_train.value_counts().rename({0: 'Healthy (0)', 1: 'Heart Disease (1)'}))

Loaded Dataset Shape: 1025 rows, 14 columns
Training Features Shape (X_train) : (820, 13)
Training Labels Shape (y_train)   : (820,)
Testing Features Shape (X_test)   : (205, 13) (Preserved untouched for Phase 4)
Testing Labels Shape (y_test)     : (205,)

Target distribution in Training Set:
target
Heart Disease (1)    421
Healthy (0)          399
Name: count, dtype: int64


## 3. Feature Standardization
Standardize feature distributions using `StandardScaler` ($\mu = 0, \sigma = 1$).  
The scaler is **fit exclusively on the training data** to avoid data leakage.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Persist the fitted scaler for inference and evaluation
scaler_path = os.path.join(MODEL_DIR, "heart_scaler.pkl")
joblib.dump(scaler, scaler_path)

print(f"[SAVED] Feature StandardScaler exported to: {scaler_path}")
print(f"Mean vector shape: {scaler.mean_.shape}")
print(f"Variance vector shape: {scaler.var_.shape}")

[SAVED] Feature StandardScaler exported to: C:\Users\Lakshya\OneDrive\Desktop\Minor-Project-1\saved_models\heart_scaler.pkl
Mean vector shape: (13,)
Variance vector shape: (13,)


## 4. Model Instantiation & Training
Initialize and fit **6 machine learning classification models**:
1. **Logistic Regression:** Linear probabilistic screening benchmark.
2. **Decision Tree (CART):** Interpretable clinical rule hierarchy.
3. **Random Forest:** Ensemble bagging to reduce variance.
4. **Support Vector Machine (RBF Kernel):** Non-linear maximum-margin hyperplane.
5. **K-Nearest Neighbors (KNN):** Distance-based non-parametric classifier.
6. **Gradient Boosting Classifier:** Sequential boosting for complex boundary mapping.

In [4]:
# 4.1 Define Model Architectures with Calibrated Hyperparameters
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=23),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=23),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=23),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=23),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=23)
}

trained_models = {}

print("=" * 75)
print(" EXECUTING MODEL TRAINING ON TRAINING COHORT (N = 820)")
print("=" * 75)

# 4.2 Fit Models on Training Data
for name, model in models.items():
    print(f"\nFitting [{name}] ...")
    
    # Distance and linear models use scaled features; tree ensembles use raw features
    if name in ['Logistic Regression', 'Support Vector Machine (RBF)', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        input_type = "Scaled Features (Standardized)"
    else:
        model.fit(X_train, y_train)
        input_type = "Raw Features (Tree-Invariant)"
        
    trained_models[name] = model
    print(f"  ✓ Training Complete. Fitted on: {input_type}")
    print(f"  ✓ Classes: {model.classes_}")

print("\n" + "=" * 75)
print(" ALL 6 HEART DISEASE MODELS SUCCESSFULLY TRAINED!")
print("=" * 75)

 EXECUTING MODEL TRAINING ON TRAINING COHORT (N = 820)

Fitting [Logistic Regression] ...
  ✓ Training Complete. Fitted on: Scaled Features (Standardized)
  ✓ Classes: [0 1]

Fitting [Decision Tree] ...
  ✓ Training Complete. Fitted on: Raw Features (Tree-Invariant)
  ✓ Classes: [0 1]

Fitting [Random Forest] ...
  ✓ Training Complete. Fitted on: Raw Features (Tree-Invariant)
  ✓ Classes: [0 1]

Fitting [Support Vector Machine (RBF)] ...


  ✓ Training Complete. Fitted on: Scaled Features (Standardized)
  ✓ Classes: [0 1]

Fitting [K-Nearest Neighbors] ...
  ✓ Training Complete. Fitted on: Scaled Features (Standardized)
  ✓ Classes: [0 1]

Fitting [Gradient Boosting] ...


  ✓ Training Complete. Fitted on: Raw Features (Tree-Invariant)
  ✓ Classes: [0 1]

 ALL 6 HEART DISEASE MODELS SUCCESSFULLY TRAINED!


## 5. Model Serialization & Persistence
Export all trained model objects to `.pkl` binaries in `saved_models/` for downstream inference and evaluation.

In [5]:
file_mapping = {
    'Logistic Regression': 'heart_logistic_regression.pkl',
    'Decision Tree': 'heart_decision_tree.pkl',
    'Random Forest': 'heart_random_forest.pkl',
    'Support Vector Machine (RBF)': 'heart_rbf_svm.pkl',
    'K-Nearest Neighbors': 'heart_knn.pkl',
    'Gradient Boosting': 'heart_gradient_boosting.pkl'
}

print("--- Serializing Trained Models to Disk ---")
saved_catalog = []

for name, model in trained_models.items():
    filename = file_mapping[name]
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    file_size_kb = os.path.getsize(save_path) / 1024
    
    saved_catalog.append({
        'Model Name': name,
        'File Name': filename,
        'File Size (KB)': f"{file_size_kb:.2f} KB",
        'Training Status': 'Trained & Serialized'
    })

catalog_df = pd.DataFrame(saved_catalog)
display(catalog_df)
print("\nAll models and scalers are persisted and ready for evaluation in Task 4.")

--- Serializing Trained Models to Disk ---


,Model Name,File Name,File Size (KB),Training Status
0,Logistic Regression,heart_logistic_regression.pkl,0.95 KB,Trained & Serialized
1,Decision Tree,heart_decision_tree.pkl,5.57 KB,Trained & Serialized
2,Random Forest,heart_random_forest.pkl,631.87 KB,Trained & Serialized
3,Support Vector Machine (RBF),heart_rbf_svm.pkl,40.53 KB,Trained & Serialized
4,K-Nearest Neighbors,heart_knn.pkl,188.04 KB,Trained & Serialized
5,Gradient Boosting,heart_gradient_boosting.pkl,131.29 KB,Trained & Serialized



All models and scalers are persisted and ready for evaluation in Task 4.
